In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
torch.manual_seed(0)
np.random.seed(0)

# ==========================================
# CONFIG
# ==========================================
HORIZON       = 60         # months (5 years)
TENORS        = 11         # yield maturities
LATENT_DIM    = 16         # ~10-20 per spec
HIDDEN_DIMS   = [128, 64]  # encoder/decoder hidden sizes (mirror)

EPOCHS        = 150
BATCH_SIZE    = 64
LR            = 1e-3

BETA_KL       = 1.0        # weight on KL term
LAMBDA_SMOOTH = 0.5        # weight on smoothness term

NUM_SCENARIOS    = 200
RAW_DATA_PATH    = "raw_data.csv"
OUTPUT_SCENARIOS = "vae_scenarios.csv"
SPLIT_DATE       = "2020-12-31"   # train cut-off; everything after is val (just for monitoring)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


# ==========================================
# 1. DATA
# ==========================================
df = pd.read_csv(RAW_DATA_PATH, index_col=0, parse_dates=True).sort_index()

# Drop macro variables explicitly
macro_cols = [c for c in ["GDP", "FedRate", "FedFunds", "CPI"] if c in df.columns]
if macro_cols:
    print(f"Dropping macro columns: {macro_cols}")
    df = df.drop(columns=macro_cols)

# Daily -> monthly EOM
df = df.resample("ME").last().dropna()

yield_cols = list(df.columns)
assert len(yield_cols) == TENORS, f"Expected {TENORS} maturities, got {len(yield_cols)}: {yield_cols}"
print("Monthly yield curve dataset:", df.shape, "from", df.index.min().date(), "to", df.index.max().date())
print("Maturities:", yield_cols)


# ==========================================
# 2. CHANGES + NORMALIZATION (training set only)
# ==========================================
df_train = df.loc[:SPLIT_DATE]
df_val   = df.loc[SPLIT_DATE:]
print(f"Train: {df_train.shape}, Val: {df_val.shape}")

# Compute monthly changes  ΔY_t = Y_t - Y_{t-1}
y_train  = df_train[yield_cols].values.astype(np.float32)
dy_train = y_train[1:] - y_train[:-1]                          # (T_train-1, 11)

y_val    = df_val[yield_cols].values.astype(np.float32)
dy_val   = y_val[1:] - y_val[:-1]

# Per-maturity normalization stats from TRAIN only
mu_dy  = dy_train.mean(axis=0)
std_dy = dy_train.std(axis=0) + 1e-8
print("Per-maturity Δy mean:", np.round(mu_dy, 5))
print("Per-maturity Δy std :", np.round(std_dy, 5))

dy_train_norm = (dy_train - mu_dy) / std_dy
dy_val_norm   = (dy_val   - mu_dy) / std_dy


# ==========================================
# 3. ROLLING WINDOWS  (60, 11)  ->  flatten to 660
# ==========================================
def make_windows(dy_norm, horizon):
    n = len(dy_norm) - horizon + 1
    if n <= 0:
        return np.empty((0, horizon, dy_norm.shape[1]), dtype=np.float32)
    return np.stack([dy_norm[i:i+horizon] for i in range(n)]).astype(np.float32)

train_windows = make_windows(dy_train_norm, HORIZON)         # (N, 60, 11)
val_windows   = make_windows(dy_val_norm,   HORIZON)
print("Train windows:", train_windows.shape, " Val windows:", val_windows.shape)

train_X = train_windows.reshape(train_windows.shape[0], -1)  # (N, 660)
val_X   = val_windows.reshape(val_windows.shape[0],   -1) if len(val_windows) else None
print("Train X (flat):", train_X.shape)


class WindowDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i]


train_loader = DataLoader(WindowDataset(train_X), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = (DataLoader(WindowDataset(val_X), batch_size=BATCH_SIZE)
                if val_X is not None and len(val_X) else None)


# ==========================================
# 4. SIMPLE MLP VAE
# ==========================================
INPUT_DIM = HORIZON * TENORS   # 660

class VAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden=HIDDEN_DIMS, latent_dim=LATENT_DIM):
        super().__init__()
        # Encoder: 660 -> 128 -> 64
        layers, prev = [], input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        self.encoder     = nn.Sequential(*layers)
        self.mu_head     = nn.Linear(prev, latent_dim)
        self.logvar_head = nn.Linear(prev, latent_dim)

        # Decoder: latent -> 64 -> 128 -> 660  (mirror)
        layers, prev = [], latent_dim
        for h in reversed(hidden):
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*layers)

    def encode(self, x):
        h = self.encoder(x)
        return self.mu_head(h), self.logvar_head(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


# ==========================================
# 5. LOSS  =  recon  +  beta * KL  +  lambda * smoothness
# ==========================================
def vae_loss(recon, target, mu, logvar, beta_kl, lambda_smooth):
    # Reconstruction (sum over features, mean over batch)
    recon_loss = ((recon - target) ** 2).sum(dim=1).mean()
    # Standard VAE KL (sum over latent, mean over batch)
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1).mean()
    # Smoothness: penalise jumps between consecutive Δy in the reconstructed path
    rec_seq = recon.view(-1, HORIZON, TENORS)
    diff = rec_seq[:, 1:, :] - rec_seq[:, :-1, :]
    smooth_loss = (diff ** 2).sum(dim=(1, 2)).mean()
    total = recon_loss + beta_kl * kl + lambda_smooth * smooth_loss
    return total, recon_loss, kl, smooth_loss


# ==========================================
# 6. TRAIN
# ==========================================
model     = VAE().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(model)

train_hist, val_hist = [], []
print("\nTraining...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    e_total, e_rec, e_kl, e_sm = [], [], [], []
    for x in train_loader:
        x = x.to(DEVICE)
        recon, mu, logvar = model(x)
        loss, rec, kl, sm = vae_loss(recon, x, mu, logvar, BETA_KL, LAMBDA_SMOOTH)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        e_total.append(loss.item()); e_rec.append(rec.item())
        e_kl.append(kl.item());      e_sm.append(sm.item())

    tr = (float(np.mean(e_total)), float(np.mean(e_rec)),
          float(np.mean(e_kl)),    float(np.mean(e_sm)))
    train_hist.append(tr)

    if val_loader is not None:
        model.eval()
        with torch.no_grad():
            v_total, v_rec, v_kl, v_sm = [], [], [], []
            for x in val_loader:
                x = x.to(DEVICE)
                recon, mu, logvar = model(x)
                loss, rec, kl, sm = vae_loss(recon, x, mu, logvar, BETA_KL, LAMBDA_SMOOTH)
                v_total.append(loss.item()); v_rec.append(rec.item())
                v_kl.append(kl.item());      v_sm.append(sm.item())
        vl = (float(np.mean(v_total)), float(np.mean(v_rec)),
              float(np.mean(v_kl)),    float(np.mean(v_sm)))
        val_hist.append(vl)

    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
        msg = (f"Ep {epoch:03d} | train tot {tr[0]:8.3f}  rec {tr[1]:8.3f}  "
               f"kl {tr[2]:7.3f}  sm {tr[3]:7.3f}")
        if val_loader is not None:
            v = val_hist[-1]
            msg += f" | val tot {v[0]:8.3f}  rec {v[1]:8.3f}"
        print(msg)


# ==========================================
# 7. LOSS CURVES
# ==========================================
train_hist_arr = np.array(train_hist)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
labels = ["Total", "Reconstruction", "KL", "Smoothness"]
for i, ax in enumerate(axes):
    ax.plot(train_hist_arr[:, i], label="train")
    if val_loader is not None and len(val_hist):
        ax.plot(np.array(val_hist)[:, i], label="val")
    ax.set_title(labels[i]); ax.set_xlabel("epoch"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


# ==========================================
# 8. GENERATE 200 SCENARIOS FROM LATEST CURVE
# ==========================================
model.eval()
with torch.no_grad():
    z = torch.randn(NUM_SCENARIOS, LATENT_DIM, device=DEVICE)
    recon = model.decode(z).cpu().numpy()                                # (N, 660)
    paths_dy_norm = recon.reshape(NUM_SCENARIOS, HORIZON, TENORS)
    # Denormalise back to raw Δy
    paths_dy = paths_dy_norm * std_dy.reshape(1, 1, -1) + mu_dy.reshape(1, 1, -1)

# Reconstruct LEVELS from latest observed yield curve
Y0         = df.iloc[-1][yield_cols].values.astype(np.float32)
start_date = df.index[-1]
print(f"\nGenerating {NUM_SCENARIOS} scenarios starting from {start_date.date()}")

paths_levels = Y0.reshape(1, 1, -1) + np.cumsum(paths_dy, axis=1)         # (N, 60, 11)

# Save scenarios CSV
scenario_dates = pd.date_range(
    start=pd.Timestamp(start_date) + pd.offsets.MonthEnd(1),
    periods=HORIZON, freq="ME",
)
rows = []
for s in range(NUM_SCENARIOS):
    for t in range(HORIZON):
        row = {"Scenario_ID": s + 1, "Month": t + 1, "Date": scenario_dates[t]}
        for i, col in enumerate(yield_cols):
            row[col] = paths_levels[s, t, i]
        rows.append(row)
scenario_df = pd.DataFrame(rows)
scenario_df.to_csv(OUTPUT_SCENARIOS, index=False)
print(f"Saved {len(scenario_df)} rows ({NUM_SCENARIOS} scenarios x {HORIZON} months) to {OUTPUT_SCENARIOS}")
print(scenario_df.head())


# ==========================================
# 9. VISUALIZE: REAL vs GENERATED
# ==========================================
def pick_tenor(name):
    if name in yield_cols: return yield_cols.index(name)
    return None

idx_short = pick_tenor("Y_DGS1")  or 0
idx_mid   = pick_tenor("Y_DGS10") or len(yield_cols) // 2
idx_long  = pick_tenor("Y_DGS30") or len(yield_cols) - 1

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (a) generated paths — short tenor
ax = axes[0, 0]
for s in range(min(40, NUM_SCENARIOS)):
    ax.plot(scenario_dates, paths_levels[s, :, idx_short], alpha=0.4, lw=1)
ax.plot([start_date], [Y0[idx_short]], "ko", label="Y_0")
ax.set_title(f"Generated scenarios — {yield_cols[idx_short]}")
ax.set_ylabel("yield (%)"); ax.grid(True, alpha=0.3); ax.legend()

# (b) generated paths — long tenor
ax = axes[0, 1]
for s in range(min(40, NUM_SCENARIOS)):
    ax.plot(scenario_dates, paths_levels[s, :, idx_long], alpha=0.4, lw=1)
ax.plot([start_date], [Y0[idx_long]], "ko", label="Y_0")
ax.set_title(f"Generated scenarios — {yield_cols[idx_long]}")
ax.set_ylabel("yield (%)"); ax.grid(True, alpha=0.3); ax.legend()

# (c) yield curve at month 60 across scenarios + Y_0
ax = axes[1, 0]
for s in range(min(60, NUM_SCENARIOS)):
    ax.plot(yield_cols, paths_levels[s, -1, :], "o-", alpha=0.25, lw=1)
ax.plot(yield_cols, Y0, "k-", lw=2.5, label="Y_0 (start)")
ax.plot(yield_cols, paths_levels[:, -1, :].mean(0), "r-", lw=2, label="gen mean @ m=60")
ax.set_title("Yield curve at month 60 (60 scenarios + start + mean)")
ax.tick_params(axis="x", rotation=45); ax.legend(); ax.grid(True, alpha=0.3)

# (d) historical mid-tenor + generated fan
ax = axes[1, 1]
hist_window = 120
ax.plot(df.index[-hist_window:], df[yield_cols[idx_mid]].values[-hist_window:],
        "k-", lw=1.5, label="historical")
mean_path = paths_levels[:, :, idx_mid].mean(axis=0)
p10 = np.percentile(paths_levels[:, :, idx_mid], 10, axis=0)
p90 = np.percentile(paths_levels[:, :, idx_mid], 90, axis=0)
ax.plot(scenario_dates, mean_path, "r-", lw=1.5, label="gen mean")
ax.fill_between(scenario_dates, p10, p90, color="r", alpha=0.2, label="10–90% band")
ax.set_title(f"{yield_cols[idx_mid]} — history + generated fan chart")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


# ==========================================
# 10. SANITY: empirical vs generated Δy stats per tenor
# ==========================================
gen_dy_full = np.diff(
    np.concatenate([np.repeat(Y0[None, None, :], NUM_SCENARIOS, axis=0), paths_levels], axis=1),
    axis=1,
)                                                                # (N, 60, 11)
gen_std_dy = gen_dy_full.reshape(-1, TENORS).std(axis=0)
emp_std_dy = dy_train.std(axis=0)

emp_corr = np.corrcoef(dy_train.T)
gen_corr = np.corrcoef(gen_dy_full.reshape(-1, TENORS).T)

stats = pd.DataFrame({
    "tenor":          yield_cols,
    "emp_std_dy":     emp_std_dy,
    "gen_std_dy":     gen_std_dy,
    "ratio_gen_emp":  gen_std_dy / emp_std_dy,
}).round(4)
print("\nPer-tenor Δy std — empirical vs generated:")
print(stats.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im0 = ax[0].imshow(emp_corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax[0].set_title("Empirical Δy correlation"); ax[0].set_xticks(range(TENORS))
ax[0].set_xticklabels(yield_cols, rotation=90); ax[0].set_yticks(range(TENORS))
ax[0].set_yticklabels(yield_cols); plt.colorbar(im0, ax=ax[0], fraction=0.046)
im1 = ax[1].imshow(gen_corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax[1].set_title("Generated Δy correlation"); ax[1].set_xticks(range(TENORS))
ax[1].set_xticklabels(yield_cols, rotation=90); ax[1].set_yticks(range(TENORS))
ax[1].set_yticklabels(yield_cols); plt.colorbar(im1, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

print(f"\nDone. Output file: {OUTPUT_SCENARIOS}  ({NUM_SCENARIOS} scenarios x {HORIZON} months x {TENORS} tenors)")
